In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
RAW = "../data/raw/"


In [4]:
sales = pd.read_excel(RAW + "Case06_Table1_Daily_SKU_Sales_by_Store.xlsx")
repl  = pd.read_excel(RAW + "Case06_Table2_Inventory_Replenishment_Log.xlsx")
sku   = pd.read_excel(RAW + "Case06_Table3_SKU_Master.xlsx")
ctx   = pd.read_excel(RAW + "Case06_Table4_Demand_Context_and_Crosswalk.xlsx", sheet_name=None)
demand_ctx = ctx["demand_context"]
crosswalk  = ctx["pincode_city_crosswalk"]

tables = {"sales": sales, "repl": repl, "sku": sku, "demand_ctx": demand_ctx, "crosswalk": crosswalk}
for name, df in tables.items():
    print(name, df.shape, list(df.columns))


sales (438000, 5) ['date', 'store_id', 'pincode', 'sku_id', 'units_sold']
repl (438000, 8) ['date', 'store_id', 'sku_id', 'on_hand_qty_open', 'order_placed_qty', 'order_received_flag', 'qty_received', 'on_hand_qty_close']
sku (80, 7) ['sku_id', 'sku_name', 'category', 'shelf_life_value', 'shelf_life_unit', 'unit_cost', 'selling_price']
demand_ctx (2190, 5) ['date', 'city', 'rainfall_mm', 'is_local_event', 'is_public_holiday']
crosswalk (15, 2) ['pincode', 'city']


In [5]:
# Cell 3: missing values in every table
for name, df in tables.items():
    miss = df.isna().sum()
    out = pd.DataFrame({"missing": miss, "pct": (miss / len(df) * 100).round(2)})
    print("\n====", name)
    print(out[out.missing > 0] if (out.missing > 0).any() else "no missing values")


==== sales
no missing values

==== repl
                     missing    pct
order_received_flag   223404  51.01

==== sku
no missing values

==== demand_ctx
no missing values

==== crosswalk
no missing values


In [6]:
# Cell 4: duplicate rows in every table
for name, df in tables.items():
    print(name, "duplicate rows:", df.duplicated().sum())

sales duplicate rows: 0
repl duplicate rows: 0
sku duplicate rows: 0
demand_ctx duplicate rows: 0
crosswalk duplicate rows: 0


In [7]:
# Cell 5: the keys must be unique (one row per date x store x SKU)
print("sales dup keys:", sales.duplicated(["date", "store_id", "sku_id"]).sum())
print("repl  dup keys:", repl.duplicated(["date", "store_id", "sku_id"]).sum())
print("sku   dup keys:", sku.duplicated(["sku_id"]).sum())

sales dup keys: 0
repl  dup keys: 0
sku   dup keys: 0


In [8]:
# Cell 6: convert dates and check ranges
sales["date"] = pd.to_datetime(sales["date"])
repl["date"] = pd.to_datetime(repl["date"])
demand_ctx["date"] = pd.to_datetime(demand_ctx["date"])

print("sales range:", sales.date.min(), "to", sales.date.max(), "| unique days:", sales.date.nunique())
print("repl  range:", repl.date.min(), "to", repl.date.max())
print("ctx   range:", demand_ctx.date.min(), "to", demand_ctx.date.max())

sales range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00 | unique days: 365
repl  range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00
ctx   range: 2024-10-01 00:00:00 to 2025-09-30 00:00:00


In [9]:
# Cell 7: impossible values
print("negative units_sold:", (sales.units_sold < 0).sum())
print("negative open stock:", (repl.on_hand_qty_open < 0).sum())
print("negative close stock:", (repl.on_hand_qty_close < 0).sum())
print("negative received:", (repl.qty_received < 0).sum())
print("selling price <= unit cost:", (sku.selling_price <= sku.unit_cost).sum())
print(sales.units_sold.describe())

negative units_sold: 0
negative open stock: 0
negative close stock: 0
negative received: 0
selling price <= unit cost: 0
count    438000.000000
mean         13.251112
std          13.650451
min           0.000000
25%           5.000000
50%          10.000000
75%          18.000000
max         490.000000
Name: units_sold, dtype: float64


In [10]:
# Cell 8: count stores, SKUs, categories, cities
print("rows in SKU master:", len(sku))
print("unique SKU ids in master:", sku.sku_id.nunique())
print("unique SKU ids in sales :", sales.sku_id.nunique())
print("unique stores in sales  :", sales.store_id.nunique())
print("SKUs in sales but not in master:", set(sales.sku_id) - set(sku.sku_id))
print("SKUs in master but not in sales:", set(sku.sku_id) - set(sales.sku_id))
print("\nSKUs per category:")
print(sku.category.value_counts())
print("\ncities:", crosswalk.city.nunique(), "| pincodes:", crosswalk.pincode.nunique())

rows in SKU master: 80
unique SKU ids in master: 80
unique SKU ids in sales : 80
unique stores in sales  : 15
SKUs in sales but not in master: set()
SKUs in master but not in sales: set()

SKUs per category:
category
Tea & Coffee        6
Rice                6
Dairy - Milk        6
Bakery - Bread      6
Leafy Vegetables    6
Spices              5
Namkeen             4
Dairy - Paneer      4
Fresh Cut Fruit     4
Root Vegetables     4
Chocolates          3
Biscuits            3
Eggs                3
Non-Veg Fresh       3
Herbs               3
Cooking Oil         2
Soap & Shampoo      2
Canned Food         2
Butter & Cheese     2
Cold Beverages      2
Dairy - Curd        2
Detergent           1
Noodles & Pasta     1
Name: count, dtype: int64

cities: 6 | pincodes: 15


In [11]:
# Cell 9: shelf life comes in two units, check the mix
print(sku.shelf_life_unit.value_counts())
print(sku.groupby("shelf_life_unit").shelf_life_value.describe())

shelf_life_unit
hours    45
days     35
Name: count, dtype: int64
                 count        mean        std   min    25%    50%    75%    max
shelf_life_unit                                                                
days              35.0  201.257143  99.922129  41.0  122.5  199.0  285.5  360.0
hours             45.0   30.888889  18.853770   6.0   14.0   27.0   45.0   68.0


In [12]:
# Cell 10: is the missing order_received_flag just "no order placed"?
repl["order_placed"] = repl.order_placed_qty > 0
print(pd.crosstab(repl.order_placed, repl.order_received_flag.fillna("blank")))

order_received_flag    N       Y   blank
order_placed                            
False                  0       0  223404
True                 340  214256       0


In [13]:
# Cell 11: orders placed but never delivered
print(repl.order_received_flag.value_counts(dropna=False))

order_received_flag
NaN    223404
Y      214256
N         340
Name: count, dtype: int64


In [14]:
# Cell 12: the 490 outlier, which rows are extreme?
print(sales.nlargest(10, "units_sold"))
print("rows above 100 units:", (sales.units_sold > 100).sum())

             date store_id  pincode   sku_id  units_sold
401359 2025-08-31     DS07   560001  SKU0080         490
333359 2025-07-05     DS12   122007  SKU0080         422
394719 2025-08-25     DS14   201307  SKU0080         404
390959 2025-08-22     DS12   122007  SKU0080         403
416239 2025-09-12     DS13   201304  SKU0080         393
304239 2025-06-11     DS08   411001  SKU0080         345
382319 2025-08-15     DS09   500004  SKU0080         344
338239 2025-07-09     DS13   201304  SKU0080         339
22159  2024-10-19     DS07   560001  SKU0080         337
433919 2025-09-27     DS09   500004  SKU0080         337
rows above 100 units: 809


In [15]:
# Cell 13: are the extreme days on one SKU or on rainy days?
big = sales[sales.units_sold > 100].merge(demand_ctx.merge(crosswalk, on="city"), on=["date", "pincode"])
print(big.groupby("sku_id").size().sort_values(ascending=False).head(10))
print(big.rainfall_mm.describe())

sku_id
SKU0080    806
SKU0043      1
SKU0050      1
SKU0059      1
dtype: int64
count    809.000000
mean      37.670087
std       27.453616
min        0.000000
25%       17.800000
50%       29.200000
75%       51.300000
max      186.900000
Name: rainfall_mm, dtype: float64


In [16]:
# Cell 14: each store maps to one pincode?
print(sales.groupby("store_id").pincode.nunique().value_counts())
print(sales[["store_id", "pincode"]].drop_duplicates().merge(crosswalk, on="pincode").sort_values("city"))

pincode
1    15
Name: count, dtype: int64
   store_id  pincode       city
6      DS07   560001  Bengaluru
9      DS10   560004  Bengaluru
14     DS15   560007  Bengaluru
0      DS01   122001    Gurgaon
4      DS05   122004    Gurgaon
11     DS12   122007    Gurgaon
5      DS06   500001  Hyderabad
8      DS09   500004  Hyderabad
2      DS03   400001     Mumbai
3      DS04   400004     Mumbai
1      DS02   201301      Noida
12     DS13   201304      Noida
13     DS14   201307      Noida
7      DS08   411001       Pune
10     DS11   411004       Pune


In [17]:
# Cell 15: make the blank flag meaningful
repl["order_received_flag"] = repl.order_received_flag.fillna("NoOrder")
print(repl.order_received_flag.value_counts())

order_received_flag
NoOrder    223404
Y          214256
N             340
Name: count, dtype: int64


In [18]:
# Cell 16: dates on which onion was actually delivered, per store
ONION = "SKU0080"
onion_in = repl[(repl.sku_id == ONION) & (repl.qty_received > 0)].sort_values(["store_id", "date"]).copy()
onion_in["gap_days"] = onion_in.groupby("store_id").date.diff().dt.days
print(onion_in.gap_days.describe())
print(onion_in.gap_days.value_counts().sort_index())

count    1397.000000
mean        3.856836
std         2.014874
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        25.000000
Name: gap_days, dtype: float64
gap_days
1.0     132
2.0     194
3.0     310
4.0     341
5.0     229
6.0      83
7.0      35
8.0      33
9.0      18
10.0      9
11.0      5
12.0      4
13.0      1
14.0      1
15.0      1
25.0      1
Name: count, dtype: int64


In [19]:
# Cell 17: gap between deliveries, per store
print(onion_in.groupby("store_id").gap_days.agg(["count", "mean", "median", "min", "max"]).round(2))

          count  mean  median  min   max
store_id                                
DS01         95  3.80     4.0  1.0  11.0
DS02         91  3.95     4.0  1.0  14.0
DS03         94  3.81     4.0  1.0   9.0
DS04         93  3.85     4.0  1.0   9.0
DS05         92  3.96     4.0  1.0  11.0
DS06         94  3.80     4.0  1.0   9.0
DS07         94  3.81     4.0  1.0  12.0
DS08         95  3.78     3.0  1.0  12.0
DS09         95  3.77     4.0  1.0  10.0
DS10         93  3.86     4.0  1.0  12.0
DS11         94  3.83     4.0  1.0  12.0
DS12         95  3.83     4.0  1.0  10.0
DS13         91  3.93     4.0  1.0  15.0
DS14         91  3.92     4.0  1.0  13.0
DS15         90  3.98     4.0  1.0  25.0


In [20]:
# Cell 18: the lag between placing an order and receiving it (order-to-arrival time)
o = repl[repl.sku_id == ONION].sort_values(["store_id", "date"]).copy()
o["placed_date"] = o.date.where(o.order_placed_qty > 0)
o["received_date"] = o.date.where(o.qty_received > 0)

# for each delivery, look back at the most recent order placed on or before that day
o["last_order_date"] = o.groupby("store_id").placed_date.ffill()
arrivals = o[o.qty_received > 0].copy()
arrivals["lead_days"] = (arrivals.date - arrivals.last_order_date).dt.days
print(arrivals.lead_days.describe())
print(arrivals.lead_days.value_counts().sort_index())

count    1412.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: lead_days, dtype: float64
lead_days
0    1412
Name: count, dtype: int64
